## 25_year_check
  Measures the effect of mixing 2021 and 2023 population: population ratios, LAU sums against
  the master table, sensitivity of b and beta. Output: results/25_year_check/*.csv.

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names, paths and years in one place) ----
# the notebook runs from notebooks/, so paths are relative to the project root one level up
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
MASTER = ROOT / "data/processed/nuts3_master_table.parquet"
LAU_MAP = ROOT / "data/processed/11_v4_model_run/lau_to_nuts3.csv"
LAU_UNITS = ROOT / "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
OUT_DIR = ROOT / "results/25_year_check"
YEAR_BASE, YEAR_NEW = 2021, 2023
LAU_POP_COL = "POP_2021"
LAU_COUNTRIES = ["HU", "SK"]  # countries where settlement-level (LAU) tables exist

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- (a) population ratio YEAR_NEW / YEAR_BASE per NUTS3 ----
# read only the needed columns (the geometry column is large)
master = pd.read_parquet(MASTER, columns=["nuts3_id", "year", "population", "country_code"])
assert not master.duplicated(["nuts3_id", "year"]).any(), "duplicate nuts3_id/year rows"

wide = (master[master["year"].isin([YEAR_BASE, YEAR_NEW])]
        .pivot(index="nuts3_id", columns="year", values="population"))
country = master.drop_duplicates("nuts3_id").set_index("nuts3_id")["country_code"]

ratio = pd.DataFrame({
    "country_code": country.reindex(wide.index),
    "pop_base": wide[YEAR_BASE],
    "pop_new": wide[YEAR_NEW],
})
ratio["ratio_new_base"] = ratio["pop_new"] / ratio["pop_base"]
n_missing = ratio[["pop_base", "pop_new"]].isna().any(axis=1).sum()
print("regions with a missing population value:", n_missing)

ratio_summary = (ratio.groupby("country_code")["ratio_new_base"]
                 .agg(n="count", median="median", min="min", max="max", std="std")
                 .round(4))
print(ratio_summary)

ratio.round(6).to_csv(OUT_DIR / "population_ratio_by_nuts3.csv")
ratio_summary.to_csv(OUT_DIR / "population_ratio_summary.csv")

# ---- (b) LAU population sum per NUTS3 vs the master table (HU, SK) ----
lau_map = pd.read_csv(LAU_MAP, dtype={"LAU_ID": str})
units = pd.read_parquet(LAU_UNITS, columns=["CNTR_CODE", "LAU_ID", LAU_POP_COL])
units["LAU_ID"] = units["LAU_ID"].astype(str)

merged = units.merge(lau_map[["CNTR_CODE", "LAU_ID", "NUTS3"]],
                     on=["CNTR_CODE", "LAU_ID"], how="left", validate="one_to_one")
print("LAU units without NUTS3:", merged["NUTS3"].isna().sum(), "of", len(merged))

lau_sum = (merged.dropna(subset=["NUTS3"])
           .groupby(["CNTR_CODE", "NUTS3"])[LAU_POP_COL].sum()
           .rename("lau_pop_sum").reset_index()
           .rename(columns={"NUTS3": "nuts3_id"}))

check = lau_sum.merge(ratio[["pop_base"]].reset_index(), on="nuts3_id", how="left")
check["ratio_lau_master"] = check["lau_pop_sum"] / check["pop_base"]
print("regions not found in master table:", check["pop_base"].isna().sum())

check_summary = (check[check["CNTR_CODE"].isin(LAU_COUNTRIES)]
                 .groupby("CNTR_CODE")["ratio_lau_master"]
                 .agg(n="count", median="median", min="min", max="max", std="std")
                 .round(4))
print(check_summary)

check.round(6).to_csv(OUT_DIR / "lau_sum_vs_master_by_nuts3.csv", index=False)
check_summary.to_csv(OUT_DIR / "lau_sum_vs_master_summary.csv")
print("saved:", *sorted(p.name for p in OUT_DIR.glob("*.csv")), sep="\n  ")

In [ ]:
import numpy as np
import statsmodels.api as sm

# ---- config ----
BETA_TS = ROOT / "results/scaling_by_country/beta_by_country_timeseries.csv"
SAMPLE = "full"
TOL = 1e-6  # tolerance of the replication check
OUT_BETA = OUT_DIR / "beta_pop_year_sensitivity.csv"

saved = pd.read_csv(BETA_TS)
saved_new = saved[(saved["year"] == YEAR_NEW) & (saved["sample"] == SAMPLE)
                  & (saved["n_variable"] == "population")].copy()

# read the master table without the geometry column
master_cols = [c for c in pd.read_parquet(MASTER).columns if c != "geometry"]
full = pd.read_parquet(MASTER, columns=master_cols)

def fit_loglog(y, x):
    """OLS of ln(y) on ln(x); returns beta, HC3 95% CI and number of regions."""
    d = pd.DataFrame({"y": y, "x": x}).dropna()
    d = d[(d["y"] > 0) & (d["x"] > 0)]
    X = sm.add_constant(np.log(d["x"]))
    res = sm.OLS(np.log(d["y"]), X).fit(cov_type="HC3")
    lo, hi = res.conf_int().iloc[1]
    return res.params.iloc[1], lo, hi, len(d)

rows = []
for _, r in saved_new.iterrows():
    ycol, grp = r["y_variable"], r["group"]
    if ycol not in full.columns:
        print("skipped (column not in master table):", ycol)
        continue
    sub = full[full["country_code"] == grp]
    if sub.empty:
        print("skipped (group is not a country code):", grp)
        continue
    y_new = sub[sub["year"] == YEAR_NEW].set_index("nuts3_id")[ycol]
    pop_new = sub[sub["year"] == YEAR_NEW].set_index("nuts3_id")["population"]
    pop_base = sub[sub["year"] == YEAR_BASE].set_index("nuts3_id")["population"]

    b_new, lo_new, hi_new, n_new = fit_loglog(y_new, pop_new)
    b_base, lo_base, hi_base, n_base = fit_loglog(y_new, pop_base)
    rows.append({
        "y_variable": ycol, "group": grp,
        "n_saved": r["n_regions"], "n_pop_new": n_new, "n_pop_base": n_base,
        "beta_saved": r["beta"], "beta_pop_new": b_new, "beta_pop_base": b_base,
        "diff_base_minus_new": b_base - b_new,
        "ci_low_new": lo_new, "ci_high_new": hi_new,
        "ci_low_base": lo_base, "ci_high_base": hi_base,
        "base_inside_new_ci": bool(lo_new <= b_base <= hi_new),
    })

out = pd.DataFrame(rows)

# replication check: refit with the 2023 population must equal the saved beta
max_diff = (out["beta_pop_new"] - out["beta_saved"]).abs().max()
print("max |refit - saved| beta:", max_diff)
assert max_diff < TOL, "replication check failed: refit differs from the saved table"

out.round(6).to_csv(OUT_BETA, index=False)
show = ["y_variable", "group", "n_pop_new", "n_pop_base", "beta_pop_new",
        "beta_pop_base", "diff_base_minus_new", "base_inside_new_ci"]
print(out[show].round(4).to_string(index=False))
print("saved:", OUT_BETA.relative_to(ROOT))

In [ ]:
# SK only: population ratio per region and the beta with each region left out in turn
sk = ratio[ratio["country_code"] == "SK"].copy()
print(sk[["pop_base", "pop_new", "ratio_new_base"]].round(4).sort_values("ratio_new_base"))

sk_ids = sk.index.tolist()
sub = full[full["country_code"] == "SK"]
y_new = sub[sub["year"] == YEAR_NEW].set_index("nuts3_id")["gdp_total"]
pop_new = sub[sub["year"] == YEAR_NEW].set_index("nuts3_id")["population"]
pop_base = sub[sub["year"] == YEAR_BASE].set_index("nuts3_id")["population"]

rows = []
for drop in [None] + sk_ids:
    keep = [i for i in sk_ids if i != drop]
    b_new = fit_loglog(y_new.loc[keep], pop_new.loc[keep])[0]
    b_base = fit_loglog(y_new.loc[keep], pop_base.loc[keep])[0]
    rows.append({"left_out": drop, "beta_pop_new": b_new, "beta_pop_base": b_base,
                 "diff_base_minus_new": b_base - b_new})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# save the SK leave-one-out table so it can be used without rerunning the notebook
sk_loo = pd.DataFrame(rows)
sk_loo.round(6).to_csv(OUT_DIR / "sk_gdp_beta_leave_one_out.csv", index=False)
print("saved:", (OUT_DIR / "sk_gdp_beta_leave_one_out.csv").relative_to(ROOT))

In [ ]:
# PROJ data fix: must run BEFORE geopandas / skmob are imported
import os
import sys
import warnings
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        candidates.append(Path(list(spec.submodule_search_locations)[0]) / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)

import numpy as np
import pandas as pd
import shapely.wkb
import geopandas as gpd
from geopandas.array import GeometryArray
import skmob
from skmob.models.gravity import Gravity

# ---- config (self-contained: this cell needs no variables from earlier cells) ----
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
MASTER = ROOT / "data/processed/nuts3_master_table.parquet"
OUT_DIR = ROOT / "results/25_year_check"
OUT_B = OUT_DIR / "b_pop_year_sensitivity.csv"
CI_CSV = ROOT / "data/processed/16_b_bootstrap/b_ci_vs_beta_ci.csv"
# positive_only: fit only the pairs with flow > 0 (PL as in 05b, FIT_INCLUDE_ZERO_PAIRS = False);
# CZ was already replicated exactly on all pairs of od_long_CZ.csv, so it stays unfiltered
COUNTRIES = {
    "CZ": {"od": ROOT / "data/processed/20_ppml_distance/od_long_CZ.csv",
           "params": ROOT / "data/processed/05_gravity_cz/gravity_params.csv",
           "positive_only": False},
    "PL": {"od": ROOT / "data/processed/20_ppml_distance/od_long_PL.csv",
           "params": ROOT / "data/processed/05b_gravity_pl/gravity_params.csv",
           "positive_only": True},
}
VARIANTS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
GRAVITY_TYPE = "singly constrained"
YEAR_BASE, YEAR_NEW = 2021, 2023
COL_ID, COL_POP, COL_COUNTRY, COL_YEAR = "nuts3_id", "population", "country_code", "year"
TILE_ID = "tile_ID"
CRS_GEO = "EPSG:4326"
TOL = 1e-4  # tolerance of the replication check

OUT_DIR.mkdir(parents=True, exist_ok=True)

# master table with the WKB geometry (same geometry in every year)
master = pd.read_parquet(MASTER, columns=[COL_ID, COL_YEAR, COL_POP, COL_COUNTRY, "geometry"])
geom_by_id = {
    r[COL_ID]: (shapely.wkb.loads(r["geometry"]) if isinstance(r["geometry"], (bytes, bytearray)) else r["geometry"])
    for _, r in master[master[COL_YEAR] == YEAR_NEW].iterrows()
}


def build_tessellation(country, year):
    """One row per region: tile id, population of the given year, geometry."""
    t = (master[(master[COL_COUNTRY] == country) & (master[COL_YEAR] == year)]
         [[COL_ID, COL_POP]].rename(columns={COL_ID: TILE_ID}).reset_index(drop=True))
    geoms = np.empty(len(t), dtype=object)
    for i, nid in enumerate(t[TILE_ID]):
        geoms[i] = geom_by_id[nid]
    return gpd.GeoDataFrame(t, geometry=GeometryArray(geoms, crs=CRS_GEO))


rows = []
for country, cfg in COUNTRIES.items():
    od = pd.read_csv(cfg["od"])[["origin", "destination", "flow"]]
    if cfg["positive_only"]:
        od = od[od["flow"] > 0]
    for year in (YEAR_NEW, YEAR_BASE):
        tess = build_tessellation(country, year)
        assert (set(od["origin"]) | set(od["destination"])) <= set(tess[TILE_ID]), "unknown region id"
        fdf = skmob.FlowDataFrame(od, origin="origin", destination="destination", flow="flow",
                                  tessellation=tess, tile_id=TILE_ID)
        for name, deterrence in VARIANTS.items():
            row = {"country": country, "model": name, "pop_year": year,
                   "n_fit_pairs": len(fdf), "status": "ok", "error_class": "",
                   "gamma": np.nan, "b": np.nan}
            # only warning class names are kept, so no path can be printed
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                try:
                    model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
                    model.fit(fdf, relevance_column=COL_POP)
                    row["gamma"] = model.deterrence_func_args[0]
                    row["b"] = model.destination_exp
                except Exception as e:
                    row["status"], row["error_class"] = "failed", type(e).__name__
            row["fit_warnings"] = ";".join(sorted({w.category.__name__ for w in caught}))
            rows.append(row)

long = pd.DataFrame(rows)
print("fit pairs per country:", long.groupby("country")["n_fit_pairs"].first().to_dict())

problems = long[(long["status"] != "ok") | (long["fit_warnings"] != "")]
if len(problems):
    print(problems[["country", "model", "pop_year", "status", "error_class", "fit_warnings"]].to_string(index=False))
else:
    print("no failed fits, no warnings")

wide = (long.pivot(index=["country", "model"], columns="pop_year", values="b")
        .rename(columns={YEAR_NEW: "b_pop_new", YEAR_BASE: "b_pop_base"}).reset_index())
wide["diff_base_minus_new"] = wide["b_pop_base"] - wide["b_pop_new"]

# replication check against the saved fits of 05 / 05b
saved = pd.concat([pd.read_csv(c["params"]).assign(country=k) for k, c in COUNTRIES.items()])
saved = saved[["country", "model", "destination_exp"]].rename(columns={"destination_exp": "b_saved"})
wide = wide.merge(saved, on=["country", "model"], how="left", validate="one_to_one")
max_diff = (wide["b_pop_new"] - wide["b_saved"]).abs().max()  # NaN if a fit failed
print("max |refit - saved| b:", max_diff)
assert max_diff < TOL, "replication check failed: the refit differs from the saved 05/05b value"

# bootstrap interval of b from notebook 16 (one row per country and model)
ci = (pd.read_csv(CI_CSV).drop_duplicates(["country", "model"])
      [["country", "model", "b_lo", "b_hi"]])
wide = wide.merge(ci, on=["country", "model"], how="left")
wide["ci_width"] = wide["b_hi"] - wide["b_lo"]
wide["diff_share_of_ci_width"] = wide["diff_base_minus_new"].abs() / wide["ci_width"]
wide["base_inside_ci"] = (wide["b_pop_base"] >= wide["b_lo"]) & (wide["b_pop_base"] <= wide["b_hi"])

wide.round(6).to_csv(OUT_B, index=False)
print(wide[["country", "model", "b_saved", "b_pop_new", "b_pop_base", "diff_base_minus_new",
            "b_lo", "b_hi", "diff_share_of_ci_width", "base_inside_ci"]].round(4).to_string(index=False))
print("saved:", OUT_B.relative_to(ROOT))

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config ----
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
CORE_DIR = ROOT / "results/core_scaling_decomposition"
FILES = ["commuting_balance_2021.csv", "core_scaling_decomposition_2021.csv", "delta_comm_2021.csv"]
SMALL_MAX_ROWS = 20  # print small tables in full, summarise larger ones

for name in FILES:
    df = pd.read_csv(CORE_DIR / name)
    print("==", name, "| rows:", len(df), "| columns:", list(df.columns))
    if len(df) <= SMALL_MAX_ROWS:
        print(df.round(4).to_string(index=False))
    else:
        print(df.describe().round(4).to_string())
    print()

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config ----
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
BALANCE_CSV = ROOT / "results/core_scaling_decomposition/commuting_balance_2021.csv"
RATIO_COL, COUNTRY_COL = "ratio_E_identity_scaled", "country_code"
TOLERANCE = 0.05  # share of regions with |ratio - 1| <= TOLERANCE

bal = pd.read_csv(BALANCE_CSV)
bal["within_tol"] = (bal[RATIO_COL] - 1).abs() <= TOLERANCE

summary = (bal.groupby(COUNTRY_COL)
           .agg(n=(RATIO_COL, "size"), mean=(RATIO_COL, "mean"),
                min=(RATIO_COL, "min"), max=(RATIO_COL, "max"),
                n_within_tol=("within_tol", "sum"))
           .round(4))
print(summary.to_string())